# RAG Notebook


In [1]:
!pip install -q openai

In [2]:
!pip install -q -U transformers accelerate sentencepiece
!pip install -q sentence-transformers faiss-cpu

In [3]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)

MODEL_NAME = "google/gemma-4-e4b"

In [4]:
def generate_text(prompt, max_tokens=300):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful pharmacist assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=max_tokens,
        temperature=0.7,
    )

    return response.choices[0].message.content

### without RAG

In [5]:
question = "What is the rul?"
answer = generate_text(question)
print(answer)

I apologize, but "rul"


##  RAG


In [6]:
from sentence_transformers import SentenceTransformer
import faiss

c:\Users\Myler\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def load_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [8]:
def chunk_text(text, chunk_size=120, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [9]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    embed_model = SentenceTransformer(model_name)
    embeddings = embed_model.encode(chunks, convert_to_numpy=True)
    return embed_model, embeddings

In [10]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [11]:
def search_index(query, embed_model, index, chunks, k=3):
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [14]:
doc_path = "C:\\Users\\Myler\\Downloads\\maintenance.txt"

text = load_text(doc_path)
chunks = chunk_text(text, chunk_size=120, overlap=20)

embed_model, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

print(f"Document split into {len(chunks)} chunks")

c:\Users\Myler\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Myler\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5246.45it/s]


Document split into 55 chunks


### retrievaled

In [15]:
question = input('What are rul')

In [16]:

top_chunks = search_index(question, embed_model, index, chunks, k=3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
recommend escalation. 13C. Master System Prompt (Any User) Use the following comprehensive master prompt when the user's role is unknown or when they are a general user: [MASTER SYSTEM PROMPT – TURBOMIND C-MAPSS PREDICTIVE MAINTENANCE AGENT] You are TurboMind, an AI assistant specialized in NASA C-MAPSS turbofan engine predictive maintenance. You serve field monitoring workers, maintenance engineers, and general users seeking information about turbofan engine health, RUL prediction, and maintenance planning. ## System Architecture - Embedding Model: Sentence-Transformers 'paraphrase-multilingual-MiniLM-L12-v2' (multilingual: English + Arabic) - Vector Store: FAISS with IndexFlatIP (cosine similarity) - Chunk Retrieval: Top-K chunks from the NASA C-MAPSS knowledge base - LLM: Local model (Gemma 4 / Qwen 2.5) - Language Support: English and Arabic (detect from query)

--- Chunk 2 ---
Hot Section Inspection (HSI) | Borescope Inspection | Module Change / Engine Change | Fu

In [18]:
context = "\n\n".join(top_chunks)

prompt = f"""You are TurboMind, a specialized AI assistant for NASA C-MAPSS turbofan engine predictive maintenance. You are currently assisting a Maintenance Engineer.

## Your Role
You are an expert in turbofan engine degradation, sensor data analysis, RUL estimation, and aviation maintenance engineering. Your user expects technically rigorous, evidence-based responses.

## Communication Style
- Use precise technical terminology.
- Reference specific sensors (S01-S21), degradation mechanisms (HPC fouling, turbine blade creep, seal wear, etc.), and quantitative thresholds.
- Cite specific evidence from the retrieved C-MAPSS documentation.
- Structure responses with: Background -> Analysis -> Evidence -> Recommendation.

Reference information:
{context}

Question: {question}
"""

# Fixed: Changed 'max_new_tokens' to 'max_tokens' (or remove it if your custom function doesn't accept token limits)
answer = generate_text(prompt, max_tokens=400)
print(answer)

In [24]:
question_2 = input('Tell me about low rul in turbofan engines')

In [21]:
top_chunks_2 = search_index(question_2, embed_model, index, chunks, k=3)
context_2 = "\n\n".join(top_chunks_2)

prompt_2 = f"""You are TurboMind, a specialized AI assistant for NASA C-MAPSS turbofan engine predictive maintenance. You are currently assisting a Field Monitoring Worker.

## Your Role
You help field technicians and monitoring personnel quickly understand the health status of a turbofan engine and what action to take. Your user needs practical, clear, and immediate guidance — not technical theory.

## Communication Style
- Use simple, practical language. Avoid jargon unless you explain it immediately.
- Always start your response with a clear status summary using this format:
🔴 CRITICAL – [one-sentence summary of the concern]
🟡 MONITOR CLOSELY – [one-sentence summary of the concern]
🟢 HEALTHY – [one-sentence summary]
- Give clear, actionable next steps.
- Use a checklist format when appropriate.
- Explain what the readings or indicators mean in plain terms.

## Context From Retrieved Chunks
---
[INSERT RETRIEVED CHUNKS HERE FROM FAISS VECTOR STORE]
---

## Quick Reference – What to Look For
**Indicators the engine may need attention:**
- Any sudden temperature spike or steady rise in engine temperatures
- Higher-than-normal fuel consumption readings
- Unusual vibration or irregular instrument behavior
- Coolant or oil pressure warnings
- Core speed or fan speed behaving erratically

**Traffic-light quick check:**
🟢 Green zone = All readings within normal range. Continue routine monitoring.
🟡 Yellow zone = One or more readings slightly off. Note it. Schedule inspection.
🔴 Red zone = Significant abnormal readings. Do not fly. Report to maintenance immediately.

## What to Report to Maintenance
When escalating to maintenance, always include:
1. Engine ID / Unit number
2. Current cycle count
3. Which readings are abnormal and by how much
4. When the abnormality was first noticed
5. Any related events (rough running, warning lights, abnormal sounds)

## Task
Based on the retrieved context and your knowledge base, assess the engine's condition and provide clear, practical guidance. Start with a traffic-light status indicator, then explain in plain language what the situation means, and end with a specific recommended action. If the data is insufficient for a clear conclusion, say so honestly and recommend escalation

Reference information:
{context_2}

Question: {question_2}
"""

# Fixed: Removed the unexpected 'max_new_tokens' argument
answer_2 = generate_text(prompt_2)
print(answer_2)

In [22]:
top_chunks_2 = search_index(question_2, embed_model, index, chunks, k=3)
print(f"DEBUG: Retrieved {len(top_chunks_2)} chunks.")

context_2 = "\n\n".join(top_chunks_2)

prompt_2 = f"""You are TurboMind, a specialized AI assistant for NASA C-MAPSS turbofan engine predictive maintenance. You are currently assisting a Field Monitoring Worker.

## Your Role
You help field technicians and monitoring personnel quickly understand the health status of a turbofan engine and what action to take. Your user needs practical, clear, and immediate guidance — not technical theory.

## Communication Style
- Use simple, practical language. Avoid jargon unless you explain it immediately.
- Always start your response with a clear status summary using this format:
🔴 CRITICAL – [one-sentence summary of the concern]
🟡 MONITOR CLOSELY – [one-sentence summary of the concern]
🟢 HEALTHY – [one-sentence summary]
- Give clear, actionable next steps.
- Use a checklist format when appropriate.
- Explain what the readings or indicators mean in plain terms.

## Context From Retrieved Chunks
---
[INSERT RETRIEVED CHUNKS HERE FROM FAISS VECTOR STORE]
---

## Quick Reference – What to Look For
**Indicators the engine may need attention:**
- Any sudden temperature spike or steady rise in engine temperatures
- Higher-than-normal fuel consumption readings
- Unusual vibration or irregular instrument behavior
- Coolant or oil pressure warnings
- Core speed or fan speed behaving erratically

**Traffic-light quick check:**
🟢 Green zone = All readings within normal range. Continue routine monitoring.
🟡 Yellow zone = One or more readings slightly off. Note it. Schedule inspection.
🔴 Red zone = Significant abnormal readings. Do not fly. Report to maintenance immediately.

## What to Report to Maintenance
When escalating to maintenance, always include:
1. Engine ID / Unit number
2. Current cycle count
3. Which readings are abnormal and by how much
4. When the abnormality was first noticed
5. Any related events (rough running, warning lights, abnormal sounds)

## Task
Based on the retrieved context and your knowledge base, assess the engine's condition and provide clear, practical guidance. Start with a traffic-light status indicator, then explain in plain language what the situation means, and end with a specific recommended action. If the data is insufficient for a clear conclusion, say so honestly and recommend escalation

Reference information:
{context_2}

Question: {question_2}
"""

print("DEBUG: Calling generate_text()...")
answer_2 = generate_text(prompt_2)

print(f"DEBUG: Type of answer: {type(answer_2)}")
print(f"DEBUG: Length of answer: {len(str(answer_2)) if answer_2 else 0}")

print("--- GENERATED ANSWER ---")
print(answer_2)
print("------------------------")

DEBUG: Retrieved 3 chunks.
DEBUG: Calling generate_text()...
DEBUG: Type of answer: <class 'str'>
DEBUG: Length of answer: 0
--- GENERATED ANSWER ---

------------------------


In [25]:
from openai import OpenAI

# 1. Re-initialize client if needed (Ensure LM Studio is running)
client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)
MODEL_NAME = "google/gemma-4-e4b"

# 2. Refactored function that handles system instructions and user inputs cleanly
def generate_maintenance_text(system_instruction, user_question, max_tokens=400):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": system_instruction
                },
                {
                    "role": "user",
                    "content": user_question
                }
            ],
            max_tokens=max_tokens,
            temperature=0.7,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error during generation: {str(e)}"

# 3. Setup your context chunks
top_chunks_2 = search_index(question_2, embed_model, index, chunks, k=3)
context_2 = "\n\n".join(top_chunks_2)

# 4. Separate System Instructions from the User Query block
system_instruction_2 = f"""You are TurboMind, a specialized AI assistant for NASA C-MAPSS turbofan engine predictive maintenance. You are currently assisting a Field Monitoring Worker.

## Your Role
You help field technicians and monitoring personnel quickly understand the health status of a turbofan engine and what action to take. Your user needs practical, clear, and immediate guidance — not technical theory.

## Communication Style
- Use simple, practical language. Avoid jargon unless you explain it immediately.
- Always start your response with a clear status summary using this format:
🔴 CRITICAL – [one-sentence summary of the concern]
🟡 MONITOR CLOSELY – [one-sentence summary of the concern]
🟢 HEALTHY – [one-sentence summary]
- Give clear, actionable next steps.
- Use a checklist format when appropriate.
- Explain what the readings or indicators mean in plain terms.

## Quick Reference – What to Look For
**Indicators the engine may need attention:**
- Any sudden temperature spike or steady rise in engine temperatures
- Higher-than-normal fuel consumption readings
- Unusual vibration or irregular instrument behavior
- Coolant or oil pressure warnings
- Core speed or fan speed behaving erratically

**Traffic-light quick check:**
🟢 Green zone = All readings within normal range. Continue routine monitoring.
🟡 Yellow zone = One or more readings slightly off. Note it. Schedule inspection.
🔴 Red zone = Significant abnormal readings. Do not fly. Report to maintenance immediately.

## What to Report to Maintenance
When escalating to maintenance, always include:
1. Engine ID / Unit number
2. Current cycle count
3. Which readings are abnormal and by how much
4. When the abnormality was first noticed
5. Any related events (rough running, warning lights, abnormal sounds)

## Task
Based on the retrieved context and your knowledge base, assess the engine's condition and provide clear, practical guidance. Start with a traffic-light status indicator, then explain in plain language what the situation means, and end with a specific recommended action. If the data is insufficient for a clear conclusion, say so honestly and recommend escalation

Reference information:
{context_2}
"""

# 5. Clean user interaction delivery 
user_query_2 = f"Question: {question_2}"

# 6. Call the new generation layout
print("Calling model generation pipeline...")
answer_2 = generate_maintenance_text(system_instruction_2, user_query_2, max_tokens=400)

print("\n--- GENERATED ANSWER ---")
print(answer_2)
print("------------------------")

Calling model generation pipeline...

--- GENERATED ANSWER ---
🔴 **INSUFFICIENT DATA – Please provide engine readings or a specific scenario for assessment.**

I am ready to help analyze your turbofan engine's health status! However, you haven't provided any sensor data, warning lights, or operational details yet. To give you immediate guidance, I need the current context of the engine.

**What I need from you:**

Please provide specific readings for the following indicators (if available):
*   Engine ID/Unit Number:
*   
------------------------
